# Kalman Filter

In [2]:
# Setup Block
from manim import *
import numpy as np

**Overview**:
<br> The Kalman Filter is an algorithm that uses a series of measurements from multiple sensors to decrease statistical noise and other inaccuracies. The algorithm was developed by Hungarian émigré Rudolf E. Kálmán, and was used very successfully in the Apollo mission with limited compute. It was also used in the U.S. Navy nuclear ballistic missile submarines, and for the navigation systems for spaceships which dock at the International Space Station. As it has been used in so many different high-stake operations, the Kalman filter is both extremely effective and efficient at handling multiple sensors and reducing error, so that is what will be used to decrease error on the robot. The Kalman filter will be calculated via an external library, so only an overview of the mathematical derivation will be outlined.
Most of this section will be directly taken from a phenomenal explanation of the filter by [Nathaniel Bubis](https://math.stackexchange.com/users/28743/nathaniel-bubis) on [Mathematics Stack Exchange.](https://math.stackexchange.com/questions/840662/an-explanation-of-the-kalman-filter) We highly recommend looking at this answer to fully understand the filter.

Figure 2. Explanation of $y_k=a_k+n_k$:

In [5]:
%%manim -qm KalmanModelExplanation

from manim import *
import numpy as np

class KalmanModelExplanation(Scene):
    def construct(self):
        banner = ManimBanner()
        banner.scale(0.3)
        banner.to_corner(DR, buff=0.2)
        
        # Define Colors for consistency
        TRUE_COLOR = GREEN
        GAIN_COLOR = ORANGE
        NOISE_COLOR = RED
        OBS_COLOR = BLUE

        # 1. Equation
        full_eq = MathTex(
            "y_k", "=", "a_k", "x_k", "+", "n_k"
        ).scale(1.5)
        full_eq.move_to(ORIGIN)

        # Labels
        label_y = Tex(r"\textbf{Observed}\\\textbf{Signal}", color=OBS_COLOR).next_to(full_eq[0], DOWN, buff=0.7).scale(0.8)
        label_a = Tex(r"\textbf{Gain}", color=GAIN_COLOR).next_to(full_eq[2], DOWN, buff=0.7).scale(0.8)
        label_x = Tex(r"\textbf{True}\\\textbf{Signal}", color=TRUE_COLOR).next_to(full_eq[3], UP, buff=0.7).scale(0.8)
        label_n = Tex(r"\textbf{Additive}\\\textbf{Noise}", color=NOISE_COLOR).next_to(full_eq[5], DOWN, buff=0.7).scale(0.8)

        # Color-code the equation parts
        full_eq[0].set_color(OBS_COLOR)
        full_eq[2].set_color(GAIN_COLOR)
        full_eq[3].set_color(TRUE_COLOR)
        full_eq[5].set_color(NOISE_COLOR)

        # --- ANIMATION PART 1: Building the Equation ---
        self.play(FadeIn(label_x), Write(full_eq[3]), banner.create())
        self.wait(0.25)
        self.play(LaggedStart(FadeIn(label_a), Write(full_eq[2]), Write(full_eq[4]), lag_ratio=0.5), run_time=0.5)
        self.wait(0.25)
        self.play(FadeIn(label_n), Write(full_eq[5]), run_time=0.5)
        self.wait(0.25)
        self.play(FadeIn(label_y), Write(full_eq[1]), Write(full_eq[0]), run_time=0.5)
        self.wait(1)

        labels = VGroup(label_y, label_a, label_x, label_n)
        self.play(FadeOut(labels))
        self.wait(0.5)

        # --- ANIMATION PART 2: Visual Breakdown ---
        self.play(full_eq.animate.shift(UP * 2).scale(0.7))

        panel_width, panel_height, buff = 3.2, 2.4, 0.3
        panels = VGroup(*[
            Rectangle(width=panel_width, height=panel_height, stroke_width=1, color=GRAY)
            for _ in range(4)
        ]).arrange(RIGHT, buff=buff).shift(DOWN * 0.5)
        
        self.play(Create(panels)) # Added this so the boxes actually show up

        def true_signal_func(t):
            return 1.5 * np.sin(0.8 * t)

        # Panel 1: True Signal
        axes1 = Axes(x_range=[-10, 10, 2], y_range=[-3, 3, 1], axis_config={"include_tip": False}).scale(0.3).move_to(panels[0])
        graph1 = axes1.plot(true_signal_func, color=TRUE_COLOR)
        desc1 = Tex(r"True State $x_k$", color=TRUE_COLOR).scale(0.6).next_to(panels[0], DOWN)
        self.play(FadeIn(axes1), Create(graph1), FadeIn(desc1))

        # Panel 2: Gain
        gain_val = 1.6
        axes2 = axes1.copy().move_to(panels[1])
        graph2 = axes2.plot(lambda t: gain_val * true_signal_func(t), color=GAIN_COLOR)
        desc2 = Tex(r"Scaled by Gain $a_k$", color=GAIN_COLOR).scale(0.6).next_to(panels[1], DOWN)
        self.play(FadeIn(axes2), Create(graph2), FadeIn(desc2))

        # Panel 3: Noise
        np.random.seed(42)
        axes3 = axes1.copy().move_to(panels[2])
        times = np.linspace(-10, 10, 100)
        noise_points = [axes3.c2p(t, 0.7 * np.random.normal(0, 1)) for t in times]
        graph3 = VMobject(color=NOISE_COLOR).set_points_as_corners(noise_points)
        desc3 = Tex(r"Random Noise $n_k$", color=NOISE_COLOR).scale(0.6).next_to(panels[2], DOWN)
        self.play(FadeIn(axes3), Create(graph3), FadeIn(desc3))

        # Panel 4: Observed
        axes4 = axes1.copy().move_to(panels[3])
        obs_points = [axes4.c2p(t, gain_val * true_signal_func(t) + 0.5 * np.random.normal(0, 1)) for t in times]
        graph4 = VMobject(color=OBS_COLOR).set_points_as_corners(obs_points)
        desc4 = Tex(r"Observed $y_k$", color=OBS_COLOR).scale(0.6).next_to(panels[3], DOWN)
        self.play(FadeIn(axes4), Create(graph4), FadeIn(desc4))
        self.wait(2)

        # --- OUTRO ---
        # Fixed: Closed the parenthesis in the line below
        self.play(
            FadeOut(VGroup(panels, axes1, axes2, axes3, graph2, graph3, desc1, desc2, desc3, desc4, full_eq)),
            graph4.animate.scale(2.5).move_to(ORIGIN), 
            axes4.animate.scale(2.5).move_to(ORIGIN), 
            graph1.animate.scale(2.5).move_to(ORIGIN)
        )

        final_summary = Tex(
            "The Kalman Filter refines estimates to minimize the ", "Error", "."
        ).scale(0.8).to_edge(UP)
        final_summary[1].set_color(RED)

        self.play(Write(final_summary))
        self.wait(2)

Manim Community v0.19.0

[03/17/26 03:17:40] INFO     Animation 0 : Using cached data (hash :                           ]8;id=643859;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=993269;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             2016333726_688922968_2358810818)                                                      

                    INFO     Animation 1 : Partial movie file written in                   ]8;id=894148;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=113863;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/home/jovyan/mainimations/media/videos/mainimations/720p30/p                         
                             artial_movie_files/KalmanModelExplanation/543634251_198510955                         
                             2_315284677.mp4'                                                                      

[03/17/26 03:17:41] INFO     Animation 2 : Partial movie file written in                   ]8;id=746364;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=4651;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/home/jovyan/mainimations/media/videos/mainimations/720p30/p                         
                             artial_movie_files/KalmanModelExplanation/543634251_384322032                         
                             3_3807933548.mp4'                                                                     

                    INFO     Animation 3 : Using cached data (hash :                           ]8;id=644120;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=412474;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             543634251_1840398133_1886498539)                                                      

...

[03/17/26 03:18:01] INFO     Animation 18 : Partial movie file written in                  ]8;id=367870;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=946521;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/home/jovyan/mainimations/media/videos/mainimations/720p30/p                         
                             artial_movie_files/KalmanModelExplanation/543634251_350239936                         
                             3_344490210.mp4'                                                                      

[03/17/26 03:18:03] INFO     Animation 19 : Partial movie file written in                  ]8;id=404186;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=927996;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/home/jovyan/mainimations/media/videos/mainimations/720p30/p                         
                             artial_movie_files/KalmanModelExplanation/543634251_424710781                         
                             4_1434180945.mp4'                                                                     

                    INFO     Combining to Movie file.                                      ]8;id=92393;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=470396;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py#739\739]8;;\

                    INFO                                                                   ]8;id=699047;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=301908;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene_file_writer.py#886\886]8;;\
                             File ready at                                                                         
                             '/home/jovyan/mainimations/media/videos/mainimations/720p30/K                         
                             almanModelExplanation.mp4'                                                            
                                                                                                                   

                    INFO     Rendered KalmanModelExplanation                                           ]8;id=796735;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=662936;file:///srv/conda/envs/notebook/lib/python3.9/site-packages/manim/scene/scene.py#255\255]8;;\
                             Played 20 animations                                                                  